[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/vae-mnist.ipynb)

# Variational Autoencoder (VAE) on Image Datasets

This notebook trains a VAE to learn a compressed representation of images, then generates new images by sampling from the learned latent space. Supports multiple datasets: MNIST, Fashion-MNIST, and CIFAR-10.

Now we'll import the necessary libraries and enable autoreload.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import matplotlib.pyplot as plt
import numpy as np
import wandb

# Import shared utilities from local package
from aiml_notebooks import create_dataset, create_dataloaders, create_trainer, plot_image_grid, log_images_to_wandb

# Enable autoreload for hot reloading
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

Next, we'll define all hyperparameters in a CONFIG dictionary and set random seeds for reproducibility.

In [ ]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Data
    'dataset_id': 'cifar10',           # Dataset: 'mnist', 'fashionmnist', or 'cifar10'
    'seed': 42,                      # Random seed for reproducibility
    'train_split': 0.857,            # 60K / 70K = ~0.857 (MNIST standard train size)
    'val_split': 0.143,              # 10K / 70K = ~0.143 (MNIST standard test size)
    'batch_size': 128,               # Number of images per batch
    'num_workers': 4,                # Parallel data loading workers
    
    # Model
    'latent_dim': 20,                # Size of compressed latent representation
    'hidden_dims': [32, 64],         # Conv layer channels (encoder/decoder)
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # Training
    'max_epochs': 50,                # Number of complete passes through data
    'log_every_n_steps': 50,         # How often to log training metrics
    
    # Loss weights
    'kl_weight': 1.0,                # Weight for KL divergence term
    
    # Visualization
    'num_samples': 16,               # Number of images to generate
    
    # Weights & Biases
    'wandb_project': 'vae-mnist',    # W&B project name
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

Now we'll load the dataset and create data loaders for training and validation.

In [ ]:
# Create dataset using the factory (handles data loading and splitting)
full_dataset, train_dataset, val_dataset = create_dataset(
    dataset_id=CONFIG['dataset_id'],
    splits=[CONFIG['train_split'], CONFIG['val_split']]
)

# Create data loaders using the factory
train_loader, val_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    use_collate_fn=False  # Vision datasets don't need padding
)

print(f"Dataset: {CONFIG['dataset_id']}")
print(f"Total samples: {len(full_dataset)}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Get a sample to check image shape
sample_batch = next(iter(train_loader))
sample_image = sample_batch[0][0]
print(f"Batch shape: {sample_batch[0].shape}")
print(f"Image shape: {sample_image.shape}")
print(f"Channels: {sample_image.shape[0]}, Height: {sample_image.shape[1]}, Width: {sample_image.shape[2]}")

Now we'll define the VAE model that automatically adapts to different image sizes and channel counts.

In [ ]:
# Variational Autoencoder with adaptive architecture for different image sizes
class VAE(L.LightningModule):
    def __init__(
        self, 
        in_channels=1,
        image_size=28,
        latent_dim=20, 
        hidden_dims=[32, 64], 
        learning_rate=1e-3, 
        kl_weight=1.0
    ):
        super().__init__()
        self.save_hyperparameters()
        
        # Calculate spatial size after convolutions
        # Two stride-2 convolutions: size -> size/2 -> size/4
        self.spatial_size = image_size // 4
        
        # Encoder: adapts to input channels and image size
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dims[0], kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_dims[0], hidden_dims[1], kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        
        # Calculate flattened size
        self.flatten_size = self.spatial_size * self.spatial_size * hidden_dims[1]
        
        # Latent space projections
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)
        
        # Decoder: latent_dim -> spatial_size x spatial_size x hidden_dims[1] -> image_size
        self.decoder_input = nn.Linear(latent_dim, self.flatten_size)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(hidden_dims[1], hidden_dims[0], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dims[0], in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),  # Output in [0, 1] range
        )
    
    def encode(self, x):
        """Encode image to latent mean and log-variance."""
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + sigma * epsilon."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        """Decode latent vector to image."""
        h = self.decoder_input(z)
        h = h.view(-1, self.hparams.hidden_dims[1], self.spatial_size, self.spatial_size)
        return self.decoder(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar
    
    def loss_function(self, recon, x, mu, logvar):
        """VAE loss = reconstruction loss + KL divergence."""
        # Reconstruction loss (binary cross-entropy)
        recon_loss = F.binary_cross_entropy(recon, x, reduction='sum')
        
        # KL divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        
        # Total loss (normalized by batch size)
        total_loss = (recon_loss + self.hparams.kl_weight * kl_loss) / x.size(0)
        
        return total_loss, recon_loss / x.size(0), kl_loss / x.size(0)
    
    def training_step(self, batch, batch_idx):
        x, _ = batch
        recon, mu, logvar = self(x)
        loss, recon_loss, kl_loss = self.loss_function(recon, x, mu, logvar)
        
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_recon_loss', recon_loss)
        self.log('train_kl_loss', kl_loss)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, _ = batch
        recon, mu, logvar = self(x)
        loss, recon_loss, kl_loss = self.loss_function(recon, x, mu, logvar)
        
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_recon_loss', recon_loss)
        self.log('val_kl_loss', kl_loss)
        
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    
    @torch.no_grad()
    def sample(self, num_samples=16):
        """Generate new images by sampling from the latent space."""
        self.eval()
        z = torch.randn(num_samples, self.hparams.latent_dim, device=self.device)
        samples = self.decode(z)
        return samples

# Initialize model with dataset-specific parameters
model = VAE(
    in_channels=sample_image.shape[0],  # 1 for grayscale, 3 for RGB
    image_size=sample_image.shape[1],   # 28 for MNIST/Fashion-MNIST, 32 for CIFAR-10
    latent_dim=CONFIG['latent_dim'],
    hidden_dims=CONFIG['hidden_dims'],
    learning_rate=CONFIG['learning_rate'],
    kl_weight=CONFIG['kl_weight'],
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input: {sample_image.shape[0]} channels, {sample_image.shape[1]}x{sample_image.shape[2]} pixels")
print(f"Latent space: {CONFIG['latent_dim']} dimensions")

Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [ ]:
# Train the model (W&B logger created automatically)
trainer = create_trainer(
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    wandb_project=CONFIG['wandb_project'],
    wandb_run_name=CONFIG['wandb_run_name'],
    wandb_config=CONFIG,
    model=model
)
trainer.fit(model, train_loader, val_loader)

Finally, we'll visualize reconstructions and generate new digits by sampling from the latent space.

In [ ]:
# Get test images for each class (0-9)
model.eval()

# Collect multiple examples of each class from validation set
class_examples = {i: [] for i in range(10)}
for images, labels in val_loader:
    for img, label in zip(images, labels):
        class_id = label.item()
        if len(class_examples[class_id]) < 100:  # Collect 100 examples per class
            class_examples[class_id].append(img)
    if all(len(examples) >= 100 for examples in class_examples.values()):
        break

# Get one representative image per class for display
test_images = torch.stack([class_examples[i][0] for i in range(10)])

# Compute mean latent vectors for each class
with torch.no_grad():
    # Encode all examples of each class to find the cluster center
    class_latent_means = []
    class_latent_stds = []
    
    for class_id in range(10):
        class_batch = torch.stack(class_examples[class_id]).to(model.device)
        mu, logvar = model.encode(class_batch)
        
        # Compute mean and std of the latent distribution for this class
        class_latent_means.append(mu.mean(dim=0))
        class_latent_stds.append(torch.exp(0.5 * logvar).mean(dim=0))
    
    class_latent_means = torch.stack(class_latent_means)
    class_latent_stds = torch.stack(class_latent_stds)
    
    # Reconstruct the representative images
    test_images_gpu = test_images.to(model.device)
    reconstructions, _, _ = model(test_images_gpu)
    
    # Generate 3 variations of each class by sampling from the class's latent cluster
    generated_rows = []
    for _ in range(3):
        # Sample from the latent space around each class's mean with its learned std
        eps = torch.randn_like(class_latent_means)
        z = class_latent_means + class_latent_stds * eps
        samples = model.decode(z)
        generated_rows.append(samples)
    
    generated = torch.cat(generated_rows, dim=0)

# Move to CPU for plotting
test_images = test_images.cpu()
reconstructions = reconstructions.cpu()
generated = generated.cpu()

# Stack all images: [Original (10) | Reconstruction (10) | Generated row 1 (10) | Generated row 2 (10) | Generated row 3 (10)]
all_images = torch.cat([test_images, reconstructions, generated], dim=0)

# Create column labels for classes 0-9
col_labels = [f'Class {i}' for i in range(10)]

# Determine colormap based on dataset
cmap = None if sample_image.shape[0] == 3 else 'gray'  # RGB uses None, grayscale uses 'gray'

# Plot using shared utility (5 rows × 10 columns)
fig = plot_image_grid(
    images=all_images,
    nrows=5,
    ncols=10,
    row_labels=['Original', 'Reconstructed', 'Generated #1', 'Generated #2', 'Generated #3'],
    col_labels=col_labels,
    figsize=(12, 6),
    cmap=cmap
)
plt.show()

# Log to W&B
log_images_to_wandb(fig, 'reconstructions')

# Dataset-specific class names
class_names = {
    'mnist': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
    'fashionmnist': ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'],
    'cifar10': ['Airplane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
}
dataset_classes = class_names.get(CONFIG['dataset_id'], [f'Class {i}' for i in range(10)])

print("\n" + "="*50)
print("VAE RESULTS")
print("="*50)
print(f"Dataset: {CONFIG['dataset_id'].upper()}")
print(f"Classes: {', '.join(dataset_classes)}")
print(f"Row 1: Original test images")
print(f"Row 2: Reconstructions (VAE output)")
print(f"Rows 3-5: Diverse samples from each class's latent space cluster")